# Supplement: Clustering

In this notebook, we examine different clustering alternatives for the candidate generation step of `NeDis`. 
1) The experiments indicate that applying `leidenalg` on correlation networks best identifies highly correlated feature modules.
2) We show the influence of the `resolution_parameter` of `leidenalg` where "higher resolutions lead to more communities, while lower resolutions lead to fewer communities" [1]. For our synthetic experiments, values around `resolution_parameter=1.3` appear to most reliably identify clusters which is why we set them as default in our implementation of `NeDis`.

> [1] Traag, V.A., Waltman, L. & van Eck, N.J. From Louvain to Leiden: guaranteeing well-connected communities. Sci Rep 9, 5233 (2019). https://doi.org/10.1038/s41598-019-41695-z

## Setup environment

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(4)

In [ ]:
import sys
import logging
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import sklearn.cluster


In [ ]:
import nedis.data.synthetic
import nedis.visualization
import nedis.cluster.leidenalg

from nedis.cluster.leidenalg import WeightedLeidenClustering
from nedis.data.synthetic import make_correlation_data_mixed
from nedis.visualization import plot_cordis_cluster

In [ ]:
logging.basicConfig(stream=sys.stdout)
logging.getLogger("nedis").setLevel(level=logging.DEBUG)

# Config

In [ ]:
# randomness
random_state = 43
np.random.seed(random_state)

In [ ]:
output_dir = pathlib.Path("../_out/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

# Data

In [ ]:
n_samples = 100
correlations = np.linspace(0.1,.9, 5)
data = [
    make_correlation_data_mixed(
        [5,10,5,5], 
        correlations=np.array([
            [1-c,0,0,0],
            [0,c,0,0],
            [0,0,1-c,-(1-c)],
            [0,0,-(1-c),1-c]]), 
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))
labels = np.repeat([0,1,2,-1], [5,10,10,15])

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 4))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    cor = correlation_matrices[yy]
    nedis.visualization.visualize_feature_clusters(cor, ax=ax, heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1))

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(
        n_components=2, random_state=random_state, init="pca")
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    ax.axis("off")
    ax.set_title(yy)
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)


fig.savefig(output_dir / "clustering_comparison_ref.png", bbox_inches="tight")
fig.savefig(output_dir / "clustering_comparison_ref.pdf", bbox_inches="tight")

## Compare clustering approaches

In [ ]:
cmap = plt.get_cmap("tab10")

In [ ]:
from matplotlib.markers import MarkerStyle

def plot_clustering_results(
        labels_true, 
        labels_pred, 
        coordinates, 
        ax=None, 
        scatter_kwargs=None):

    if ax is None:
        ax = plt.gca()

    labels_true_unique = np.unique(labels_true)
    labels_pred_unique = np.unique(labels_pred)

    label_map = {}
    colors = {}
    # assign true labels to the most similar predicted labels
    for i, label_true in enumerate(labels_true_unique):
        overlapping_labels_pred, counts = np.unique(
            labels_pred[labels_true == label_true], 
            return_counts=True)
        most_frequent_label_pred = overlapping_labels_pred[np.argmax(counts)]
        label_map[most_frequent_label_pred] = label_true
        colors[label_true] = cmap(i) if label_true >= 0 else (0.7,0.7,0.7,1.0)

    # assign new labels to the remaining predicted labels 
    # so they don't overlap with the true labels
    max_label_true = labels_true_unique.max()
    labels_pred_unique_remaining = [l for l in labels_pred_unique if l not in label_map]
    for i, label_pred in enumerate(labels_pred_unique_remaining):
        new_label = max_label_true + 1
        label_map[label_pred] = new_label
        # colors[new_label] = cmap(len(labels_true_unique) + i)

        colors[new_label] = (0.9,0.9,.9,1.0)
        max_label_true += 1

    labels_pred_mapped = np.array([label_map[l] for l in labels_pred])
    colors_true = np.array([colors[l] for l in labels_true])
    colors_pred = np.array([colors[l] for l in labels_pred_mapped])

    scatter_kwargs_default = {}
    scatter_kwargs = \
        scatter_kwargs_default if scatter_kwargs is None \
        else {**scatter_kwargs_default, ** scatter_kwargs}
    
    msk_correct = labels_true == labels_pred_mapped
    msk_missed = labels_true != labels_pred_mapped

    # map labels
    ax.scatter(
        coordinates[msk_correct,0], 
        coordinates[msk_correct,1], 
        facecolors=colors_true[msk_correct],
        # facecolors="none",
        # edgecolors=colors_true,
        # linewidths=7,
        s=800,
        **scatter_kwargs)
    
    ax.scatter(
        coordinates[msk_missed,0], 
        coordinates[msk_missed,1], 
        marker=MarkerStyle('o', fillstyle='left'),
        facecolors=colors_true[msk_missed],
        edgecolors="white",
        linewidths=2,
        s=1200,
        **scatter_kwargs)
    
    ax.scatter(
        coordinates[msk_missed,0], 
        coordinates[msk_missed,1], 
        marker=MarkerStyle('o', fillstyle='right'),
        facecolors=colors_pred[msk_missed],
        edgecolors="white",
        linewidths=2,
        s=1200,
        **scatter_kwargs)


In [ ]:
from tkinter import font
from sklearn.cluster import DBSCAN, KMeans
from sklearn.base import BaseEstimator


clustering_algorithms = {
    "B1. Ground Truth": labels,
    "B2. Leidenalg (r=1.3)": WeightedLeidenClustering(resolution_parameter=1.3),
    "B3. KMeans (k=4)": KMeans(n_clusters=4),
    "B4. DBSCAN (eps=2, min=3)": DBSCAN(eps=2, min_samples=3),     
} 

# more space between rows
fig, axes = plt.subplots(
    len(clustering_algorithms), 
    len(y_unique), 
    figsize=(4 * 2 * len(y_unique), 4 * 2 * len(clustering_algorithms)))
fig.tight_layout(h_pad=10, w_pad=0)

for i_row, (name, clustering) in enumerate(clustering_algorithms.items()):
    ax = axes[i_row, 0]
    ax.set_title(
        name, 
        fontdict={"fontsize": 42, "ha": "left", "va": "center"}, 
        pad=20, 
        loc="left"
        , y=1.07)
    for i_col, yy in enumerate(y_unique):
        ax = axes[i_row, i_col]
        ax.axis("off")
        emb = coordinates[y_unique[0]]
        plot_clustering_results(
            labels, 
            clustering.fit_predict(correlation_matrices[yy]) 
                if isinstance(clustering, BaseEstimator) 
                else clustering, 
            emb, 
            ax=ax)

fig.savefig(output_dir / "clustering_comparison.png", bbox_inches="tight")
fig.savefig(output_dir / "clustering_comparison.pdf", bbox_inches="tight")


## Compare `leidenalg` resolution parameters

In [ ]:
from tkinter import font
from sklearn.cluster import DBSCAN, KMeans


clustering_algorithms = {

    "Leidenalg (r=0.5)": WeightedLeidenClustering(resolution_parameter=0.5),
    "Leidenalg (r=1.0)": WeightedLeidenClustering(resolution_parameter=1.0),
    "Leidenalg (r=1.3)": WeightedLeidenClustering(resolution_parameter=1.3),
    "Leidenalg (r=2.0)": WeightedLeidenClustering(resolution_parameter=2.0),
    "Leidenalg (r=5.0)": WeightedLeidenClustering(resolution_parameter=5.0),

}

# more space between rows
fig, axes = plt.subplots(
    len(clustering_algorithms), 
    len(y_unique), 
    figsize=(4 * 2 * len(y_unique), 4 * 2 * len(clustering_algorithms)))
fig.tight_layout(h_pad=10, w_pad=0)

for i_row, (name, clustering) in enumerate(clustering_algorithms.items()):
    ax = axes[i_row, 0]
    ax.set_title(
        name, 
        fontdict={"fontsize": 42, "ha": "left", "va": "center"}, 
        pad=20, 
        loc="left"
        , y=1.07)
    for i_col, yy in enumerate(y_unique):
        ax = axes[i_row, i_col]
        ax.axis("off")
        emb = coordinates[y_unique[0]]
        plot_clustering_results(labels, clustering.fit_predict(correlation_matrices[yy]), emb, ax=ax)

fig.savefig(output_dir / "clustering_resolution_params.png", bbox_inches="tight")
fig.savefig(output_dir / "clustering_resolution_params.pdf", bbox_inches="tight")
